# Strategy Development Laboratory

Copy this notebook, rename it (e.g. `lighter_c_bbo_v2.ipynb`), and develop your strategy.

**Workflow:** Load data → Define strategy → Test

The strategy name is automatically derived from the notebook name. When you edit the strategy cell and run it, the file updates automatically.

In [1]:
import json
from pathlib import Path
import os
import numpy as np
import pandas as pd
from leadlag import list_sessions, load_session, Strategy, Order, Context, Event, BboSnapshot

DATA = '../data'
sessions = list_sessions(DATA)
session = load_session(sessions[0]['id'], data_dir=DATA)
events = session.events.filter(signal='C')

print(f"Session: {session.session_id}")
print(f"Duration: {session.meta['duration_s']:.0f}s  |  Events: {session.events.count}  |  Signal C: {events.count}")
print(f"Followers: {session.meta['followers']}")
fees_preview = ', '.join(
    f"{v}: {session.meta['fees'][v]['taker_bps']}"
    for v in session.meta['followers'][:3]
)
print(f"Fees (taker bps): {fees_preview}...")
bbo_available = [v for v, ok in session.meta.get('bbo_available', {}).items() if ok][:5]
print(f"BBO available: {bbo_available}...")

Session: 20260419_080345_d35a5f61
Duration: 14500s  |  Events: 355  |  Signal C: 132
Followers: ['Binance Perp', 'Binance Spot', 'Bitget Perp', 'Bybit Spot', 'Gate Perp', 'Hyperliquid Perp', 'Lighter Perp', 'MEXC Perp', 'edgeX Perp']
Fees (taker bps): Binance Perp: 4.5, Binance Spot: 10.0, Bitget Perp: 6.0...
BBO available: ['OKX Perp', 'Bybit Perp', 'Binance Perp', 'Binance Spot', 'Bitget Perp']...


In [2]:
import json
import os
from pathlib import Path
from urllib.request import Request, urlopen

from ipykernel import get_connection_file


def get_notebook_name(default: str = "strategy") -> str:
    try:
        connection_file = Path(get_connection_file()).name
        kernel_id = connection_file.replace("kernel-", "").replace(".json", "")

        runtime_dir = Path(
            os.environ.get("JUPYTER_RUNTIME_DIR", "~/.local/share/jupyter/runtime")
        ).expanduser()

        for server_file in list(runtime_dir.glob("jpserver-*.json")) + list(runtime_dir.glob("nbserver-*.json")):
            try:
                server = json.loads(server_file.read_text())
            except Exception:
                continue

            url = server.get("url")
            token = server.get("token", "")
            if not url:
                continue

            sessions_url = url.rstrip("/") + "/api/sessions"
            if token:
                sep = "&" if "?" in sessions_url else "?"
                sessions_url = f"{sessions_url}{sep}token={token}"

            try:
                req = Request(sessions_url)
                with urlopen(req, timeout=2) as resp:
                    sessions = json.loads(resp.read().decode("utf-8"))
            except Exception:
                continue

            for session in sessions:
                kernel = session.get("kernel", {})
                if kernel.get("id") != kernel_id:
                    continue

                notebook = session.get("notebook", {})
                path = notebook.get("path") or notebook.get("name")
                if path:
                    return Path(path).stem

        return default

    except Exception:
        return default


STRATEGY_NAME = get_notebook_name()
print(f"Strategy will be saved as: {STRATEGY_NAME}")


Strategy will be saved as: strategy_dev


In [3]:
%%writefile ../data/strategies/{STRATEGY_NAME}.py
from leadlag import Strategy, Order


class MyStrategy(Strategy):
    version = "2026-04-18"
    description = "Strategy template"
    params = {
        "signal": "C",
        "follower": "Lighter Perp",
        "min_magnitude": 2.0,
        "hold_ms": 30000,
    }

    def on_event(self, event, ctx):
        p = self.params

        if event.signal != p["signal"]:
            return None
        if p["follower"] not in event.lagging_followers:
            return None
        if event.magnitude_sigma < p["min_magnitude"]:
            return None

        return Order(
            venue=p["follower"],
            side="buy" if event.direction > 0 else "sell",
            qty_btc=0.001,
            entry_type="market",
            hold_ms=p["hold_ms"],
        )

Overwriting ../data/strategies/strategy_dev.py


In [4]:
from leadlag import load_strategy

strat = load_strategy(f'../data/strategies/{STRATEGY_NAME}.py')
print(f"Loaded: {strat.name}")
print(f"Params: {strat.params}")

# Quick test with mock event
event = Event(
    bin_idx=0, ts_ms=0, signal='C', direction=1,
    magnitude_sigma=3.0, leader='OKX Perp', lagging_followers=['Lighter Perp']
)
ctx = Context(ts_ms=0, bbo={'Lighter Perp': BboSnapshot('Lighter Perp', spread_bps=1.5)})
order = strat.on_event(event, ctx)
print(f"\nTest order: {order}")

Loaded: UnnamedStrategy
Params: {'signal': 'C', 'follower': 'Lighter Perp', 'min_magnitude': 2.0, 'hold_ms': 30000}

Test order: Order(venue='Lighter Perp', side='buy', entry_type='market', limit_price=None, hold_ms=30000, delay_ms=0, stop_loss_bps=None, take_profit_bps=None, qty_btc=0.001, tag=None)


In [5]:
# Test with params variations
print("Test 1: Change min_magnitude")
strat.params['min_magnitude'] = 5.0
order = strat.on_event(event, ctx)
print(f"  Order (stricter): {order}")

print("\nTest 2: Change follower")
strat.params['follower'] = 'Binance Perp'
event.lagging_followers = ['Binance Perp']
order = strat.on_event(event, ctx)
print(f"  Order (other follower): {order}")

# Reset
strat.params['min_magnitude'] = 2.0
strat.params['follower'] = 'Lighter Perp'
event.lagging_followers = ['Lighter Perp']

Test 1: Change min_magnitude
  Order (stricter): None

Test 2: Change follower
  Order (other follower): None
